In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("mammo-bench_master_split.csv")
counts = df['birads'].value_counts()
print(counts, counts.sum()) 

counts = df['classification'].value_counts()
print(counts, counts.sum()) 

counts = df['density'].value_counts()
print(counts, counts.sum()) 

counts = df['abnormality'].value_counts()
print(counts, counts.sum()) 

counts = df['molecular_subtype'].value_counts()
print(counts, counts.sum()) 

birads
1.0    17759
2.0     5063
0.0     4659
3.0     1082
4.0     1000
5.0      692
Name: count, dtype: int64 30255
classification
Normal       22239
Benign        7250
Unknown       3291
Malignant     2703
Name: count, dtype: int64 35483
density
C    15722
B    10632
A     3141
D     2691
Name: count, dtype: int64 32186
abnormality
mass             2624
calcification    1204
normal            200
Name: count, dtype: int64 4028
molecular_subtype
Luminal B          507
Luminal A          224
triple negative    143
HER2-enriched      143
Name: count, dtype: int64 1017


In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("mammo-bench_master_split.csv")
train_df = df[df['split'] == 'train']
counts = train_df['birads'].value_counts().sort_index()
# Inverse frequency weighting, normalized to median
weights = counts.median() / counts
weights = weights.clip(upper=10.0)  # cap extreme weights
print(weights.round(2))

birads
0.0    0.62
1.0    0.16
2.0    0.57
3.0    2.67
4.0    2.92
5.0    4.17
Name: count, dtype: float64


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("mammo-bench_master_split.csv")
train_df = df[df['split'] == 'train']
# filter out the unknown class
train_df = train_df[train_df['classification'] != 'Unknown']
counts = train_df['classification'].value_counts().sort_index()
# Inverse frequency weighting, normalized to median
weights = counts.median() / counts
weights = weights.clip(upper=10.0)  # cap extreme weights
print(weights.round(2))

classification
Benign       1.00
Malignant    2.68
Normal       0.32
Name: count, dtype: float64


In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("mammo-bench_master_split.csv")
train_df = df[df['split'] == 'train']
counts = train_df['density'].value_counts().sort_index()
# Inverse frequency weighting, normalized to median
weights = counts.median() / counts
weights = weights.clip(upper=10.0)  # cap extreme weights
print(weights.round(2))

density
A    2.19
B    0.65
C    0.44
D    2.57
Name: count, dtype: float64


In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv("mammo-bench_master_split.csv")
train_df = df[df['split'] == 'train']
counts = train_df['abnormality'].value_counts().sort_index()
# Inverse frequency weighting, normalized to median
weights = counts.median() / counts
weights = weights.clip(upper=10.0)  # cap extreme weights
print(weights.round(2))

abnormality
calcification    1.00
mass             0.46
normal           5.99
Name: count, dtype: float64


In [5]:
import pandas as pd
import numpy as np

df = pd.read_csv("mammo-bench_master_split.csv")
train_df = df[df['split'] == 'train']
counts = train_df['molecular_subtype'].value_counts().sort_index()
# Inverse frequency weighting, normalized to median
weights = counts.median() / counts
weights = weights.clip(upper=10.0)  # cap extreme weights
print(weights.round(2))

molecular_subtype
HER2-enriched      1.26
Luminal A          0.83
Luminal B          0.36
triple negative    1.33
Name: count, dtype: float64


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("mammo-bench_master_split.csv")
test_df = df[df["split"] == "test"]

for dataset in ["rsna-screening", "ibia", "inbreast", "cdd-cesm"]:
    sub = test_df[test_df["source_dataset"] == dataset]
    print(f"\n{dataset} (n={len(sub)})")
    print("classification values:", sub["classification"].value_counts().to_dict())
    print("birads values:", sub["birads"].value_counts().to_dict())


rsna-screening (n=2681)
classification values: {'Normal': 1997, 'Unknown': 503, 'Benign': 181}
birads values: {1.0: 1308, 0.0: 689, 2.0: 181}

ibia (n=311)
classification values: {'Benign': 180, 'Normal': 101, 'Malignant': 30}
birads values: {2.0: 167, 1.0: 101, 5.0: 21, 3.0: 12, 4.0: 10}

inbreast (n=51)
classification values: {'Benign': 38, 'Normal': 7, 'Malignant': 6}
birads values: {2.0: 34, 1.0: 7, 5.0: 6, 3.0: 4}

cdd-cesm (n=94)
classification values: {'Normal': 37, 'Malignant': 31, 'Benign': 26}
birads values: {1.0: 45, 5.0: 19, 2.0: 12, 4.0: 11, 3.0: 6}


In [4]:
preds = pd.read_csv("test_results_unc_wt_ep49/predictions_test.csv")
rsna_preds = preds[preds["source_dataset"] == "rsna-screening"]
print(rsna_preds["classification_pred"].value_counts())
print(rsna_preds["classification_pred"].describe())  # Normal prob distribution

classification_pred
0    2564
1     116
2       1
Name: count, dtype: int64
count    2681.000000
mean        0.044013
std         0.206974
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         2.000000
Name: classification_pred, dtype: float64


In [5]:
"""
compute_dataset_stats.py

Computes per-dataset mean and std from the training split.
Run once, paste output into dataset.py.

Run: python compute_dataset_stats.py
"""

import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

CSV_PATH   = "mammo-bench_master_split.csv"
IMAGE_ROOT = "/data/gaurav.bhole/FullDataset/"
IMG_SIZE   = 224
MAX_PER_DATASET = 500   # cap to keep runtime reasonable; increase for more accuracy

df = pd.read_csv(CSV_PATH)
train_df = df[df["split"] == "train"].copy()

datasets = train_df["source_dataset"].unique()

results = {}

for dataset in sorted(datasets):
    sub = train_df[train_df["source_dataset"] == dataset]
    
    # collect all available image paths
    paths = []
    for col in ["cc_path", "mlo_path"]:
        valid = sub[col].dropna().tolist()
        paths.extend(valid)
    
    paths = list(set(paths))  # deduplicate
    if len(paths) > MAX_PER_DATASET:
        rng = np.random.default_rng(42)
        paths = rng.choice(paths, MAX_PER_DATASET, replace=False).tolist()

    means, stds = [], []
    skipped = 0

    for path in tqdm(paths, desc=dataset, leave=False):
        full = os.path.join(IMAGE_ROOT, path) if not path.startswith("/") else path
        if not os.path.exists(full):
            skipped += 1
            continue
        img = cv2.imread(full)
        if img is None:
            skipped += 1
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
        t = img.astype(np.float32) / 255.0   # HWC, [0,1]
        means.append(t.mean(axis=(0, 1)))     # per-channel mean
        stds.append(t.std(axis=(0, 1)))       # per-channel std

    if not means:
        print(f"  WARNING: No images loaded for {dataset}, skipping")
        continue

    mean = np.stack(means).mean(axis=0)
    std  = np.stack(stds).mean(axis=0)

    results[dataset] = {
        "mean": mean.tolist(),
        "std":  std.tolist(),
        "n_images": len(means),
        "n_skipped": skipped,
    }

    print(f"\n{dataset}  (n={len(means)}, skipped={skipped})")
    print(f"  mean: [{mean[0]:.4f}, {mean[1]:.4f}, {mean[2]:.4f}]")
    print(f"  std:  [{std[0]:.4f}, {std[1]:.4f}, {std[2]:.4f}]")

# ── print as copy-pasteable dict for dataset.py ──────────────────────────
print("\n\n" + "=" * 60)
print("# Paste this into dataset.py:")
print("DATASET_STATS = {")
for ds, stats in results.items():
    m = stats["mean"]
    s = stats["std"]
    print(f'    "{ds}": {{')
    print(f'        "mean": [{m[0]:.4f}, {m[1]:.4f}, {m[2]:.4f}],')
    print(f'        "std":  [{s[0]:.4f}, {s[1]:.4f}, {s[2]:.4f}],')
    print(f'    }},')
# fallback = average across all datasets
all_means = np.array([v["mean"] for v in results.values()])
all_stds  = np.array([v["std"]  for v in results.values()])
gm = all_means.mean(axis=0)
gs = all_stds.mean(axis=0)
print(f'    "_default": {{')
print(f'        "mean": [{gm[0]:.4f}, {gm[1]:.4f}, {gm[2]:.4f}],')
print(f'        "std":  [{gs[0]:.4f}, {gs[1]:.4f}, {gs[2]:.4f}],')
print(f'    }},')
print("}")


cbis-ddsm  (n=500, skipped=0)
  mean: [0.1690, 0.1690, 0.1690]
  std:  [0.2327, 0.2327, 0.2327]



cdd-cesm  (n=500, skipped=0)
  mean: [0.2058, 0.2058, 0.2058]
  std:  [0.1866, 0.1866, 0.1866]



cmmd  (n=500, skipped=0)
  mean: [0.0599, 0.0599, 0.0599]
  std:  [0.1338, 0.1338, 0.1338]



dmid  (n=282, skipped=0)
  mean: [0.1395, 0.1395, 0.1395]
  std:  [0.2312, 0.2312, 0.2312]



ibia  (n=500, skipped=0)
  mean: [0.1013, 0.1013, 0.1013]
  std:  [0.1692, 0.1692, 0.1692]



inbreast  (n=274, skipped=0)
  mean: [0.1600, 0.1600, 0.1600]
  std:  [0.2470, 0.2470, 0.2470]



kau-bcmd  (n=500, skipped=0)
  mean: [0.0764, 0.0764, 0.0764]
  std:  [0.1449, 0.1449, 0.1449]



rsna-screening  (n=500, skipped=0)
  mean: [0.2326, 0.2326, 0.2326]
  std:  [0.2002, 0.2002, 0.2002]



vindr-mammo  (n=500, skipped=0)
  mean: [0.2288, 0.2288, 0.2288]
  std:  [0.2267, 0.2267, 0.2267]


# Paste this into dataset.py:
DATASET_STATS = {
    "cbis-ddsm": {
        "mean": [0.1690, 0.1690, 0.1690],
        "std":  [0.2327, 0.2327, 0.2327],
    },
    "cdd-cesm": {
        "mean": [0.2058, 0.2058, 0.2058],
        "std":  [0.1866, 0.1866, 0.1866],
    },
    "cmmd": {
        "mean": [0.0599, 0.0599, 0.0599],
        "std":  [0.1338, 0.1338, 0.1338],
    },
    "dmid": {
        "mean": [0.1395, 0.1395, 0.1395],
        "std":  [0.2312, 0.2312, 0.2312],
    },
    "ibia": {
        "mean": [0.1013, 0.1013, 0.1013],
        "std":  [0.1692, 0.1692, 0.1692],
    },
    "inbreast": {
        "mean": [0.1600, 0.1600, 0.1600],
        "std":  [0.2470, 0.2470, 0.2470],
    },
    "kau-bcmd": {
        "mean": [0.0764, 0.0764, 0.0764],
        "std":  [0.1449, 0.1449, 0.1449],
    },
    "rsna-screening": {
        "mean": [0.2326, 0.2326, 0.2326],
        "std":  [0.2002, 0.2002, 

In [1]:
import torch
from configs.config import Config
from models.model import MammoSightModel

# ── Config ────────────────────────────────────────────────────────────────
cfg = Config()
CHECKPOINT = "/data/gaurav.bhole/MammoSight_unc_wt_checkpoints/checkpoint_ep49.pth"  # <-- change this

# ── Load model ────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MammoSightModel(
    backbone_type=cfg.BACKBONE,
    backbone_name=cfg.BACKBONE_NAME,
    sam_checkpoint_path=cfg.SAM_CHECKPOINT,
    pooling_mode=cfg.POOLING_MODE,
    fusion_mode=cfg.FUSION_MODE,
).to(device)

ckpt = torch.load(CHECKPOINT, map_location=device)
model.load_state_dict(ckpt, strict=False)
model.eval()

# ── Total counts ──────────────────────────────────────────────────────────
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen    = total - trainable

print(f"{'Total parameters:':<25} {total:>15,}  ({total/1e6:.2f}M)")
print(f"{'Trainable parameters:':<25} {trainable:>15,}  ({trainable/1e6:.2f}M)")
print(f"{'Frozen parameters:':<25} {frozen:>15,}  ({frozen/1e6:.2f}M)")

# ── Per-component breakdown ───────────────────────────────────────────────
print("\n── Per-module breakdown ──────────────────────────────────────────")
print(f"{'Module':<35} {'Total':>10}  {'Trainable':>10}  {'Frozen':>10}")
print("-" * 70)
for name, module in model.named_children():
    t = sum(p.numel() for p in module.parameters())
    tr = sum(p.numel() for p in module.parameters() if p.requires_grad)
    fr = t - tr
    print(f"{name:<35} {t/1e6:>9.2f}M  {tr/1e6:>9.2f}M  {fr/1e6:>9.2f}M")

⚠️ Warning: Failed to load local tokenizer from /data/gaurav.bhole/pretrained_models/medimageinsights/2024.09.27/language_model/clip_tokenizer_4.16.2. Falling back to default OpenAI CLIP.
⚠️ Warning: Failed to load local tokenizer from /data/gaurav.bhole/pretrained_models/medimageinsights/2024.09.27/language_model/clip_tokenizer_4.16.2. Falling back to default OpenAI CLIP.
Model loaded successfully on device: cuda:1
[model] backbone=medimageinsight  embed_dim=2048  pooling=sam_roi  fusion=cross_attention
[model] Loaded SAM decoder weights (120 keys)
Total parameters:             367,416,821  (367.42M)
Trainable parameters:         367,416,821  (367.42M)
Frozen parameters:                      0  (0.00M)

── Per-module breakdown ──────────────────────────────────────────
Module                                   Total   Trainable      Frozen
----------------------------------------------------------------------
backbone                               360.64M     360.64M       0.00M
adapte